# Guía de pruebas Fixify — entrega RF01–RF04

Actualización: **05-Oct-2026**. Ejecutar desde la raíz del repositorio, dentro del entorno Python del proyecto. Las celdas usan el intérprete del kernel y no ejecutan pruebas al abrir el notebook.

## Preparación

```bash
python -m venv .venv
# Linux
source .venv/bin/activate
python -m pip install -r requirements-linux.txt
# Windows (PowerShell)
.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
```

En Linux instalar `python3-tkinter` con el gestor del sistema (`sudo dnf install python3-tkinter` en Fedora; `sudo apt install python3-tk` en Debian/Ubuntu). Tkinter no se instala con pip. Para pruebas Python no hace falta PostgreSQL ni credenciales remotas: los casos backend configuran SQLite y una clave de pruebas. Las pruebas frontend usan transportes HTTP simulados; las del backend también prueban el servicio HTTP contra el router real.

Un entorno que bloquea hilos/sockets puede bloquear TestClient o el runner. Ejecutar las pruebas desde una terminal local normal. Una interfaz gráfica solo hace falta para la comprobación manual.


## Ruta de ejecución: desde preparar el entorno hasta repetir un caso

Esta sección desarrolla los comandos de las instrucciones anteriores. **Elegir Linux o Windows; no ejecutar ambas preparaciones.** La raíz del proyecto es la carpeta que contiene `backend`, `frontend`, `docs` y los requirements. Abrir la terminal del IDE en esa carpeta. Los comandos con rutas relativas se interpretan desde la carpeta actual.

### 1. Crear un entorno aislado

```bash
python -m venv .venv
```

| Parte | Qué hace | Por qué se usa |
| --- | --- | --- |
| `python` | Ejecuta el intérprete que resuelve la terminal | Define la versión de Python que tendrá el entorno. En Linux puede llamarse `python3`; en Windows puede usarse `py` si ese es el lanzador disponible |
| `-m` | Ejecuta un módulo de Python como programa | Permite usar herramientas incluidas en ese mismo intérprete |
| `venv` | Módulo de la biblioteca estándar que crea entornos virtuales | Aísla las dependencias de Fixify de otros proyectos |
| `.venv` | Carpeta de destino, relativa a la raíz | Guarda el intérprete y paquetes del entorno; el nombre es una convención, no una opción especial |

Ejecutar la creación una vez si todavía no existe el entorno. Si el comando disponible es `python3`, usar `python3 -m venv .venv`. Si se usa `py`, usar `py -m venv .venv`. Después de activar el entorno, los siguientes ejemplos usan `python`.

### 2. Activar el entorno de la terminal

**Linux, Bash:**

```bash
source .venv/bin/activate
```

`source` hace que Bash ejecute el script en la terminal actual. Así modifica `PATH` y otras variables para que `python` y `pip` apunten al entorno. `.venv/bin/activate` es la ruta del script generado por venv. Ejecutarlo como un proceso separado no actualizaría la terminal padre.

**Windows 11, PowerShell:**

```powershell
.\.venv\Scripts\Activate.ps1
```

`./` escrito como `.\` indica una ruta desde la carpeta actual; PowerShell requiere una ruta explícita para invocar este script local. `Scripts` es la carpeta de ejecutables del venv en Windows y `Activate.ps1` es el script de activación para PowerShell. No es un comando de Bash ni del símbolo del sistema CMD.

La activación afecta solamente a esa terminal. Si PowerShell bloquea el script o se quiere evitar activarlo, usar directamente el intérprete del entorno:

```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
.\.venv\Scripts\python.exe -m unittest discover -s backend/tests -v
```

En Linux también se puede usar directamente `.venv/bin/python`. Así no es necesario cambiar políticas de PowerShell ni depender de la activación.

Comprobar el intérprete antes de instalar o ejecutar:

```bash
python -c "import sys; print(sys.executable); print(sys.version)"
```

`-c` ejecuta el código Python escrito en el argumento entre comillas. `sys.executable` muestra el ejecutable utilizado y `sys.version` su versión. Debe corresponder al entorno elegido. Las comillas mantienen todo el código como un argumento; este comando no ejecuta pruebas.

### 3. Instalar las dependencias del proyecto

**Linux:**

```bash
python -m pip install -r requirements-linux.txt
```

**Windows:**

```powershell
python -m pip install -r requirements.txt
```

| Parte | Qué hace | Por qué se usa |
| --- | --- | --- |
| `python -m pip` | Ejecuta pip con ese Python | Evita instalar paquetes en otro intérprete por usar un `pip` distinto en PATH |
| `install` | Subcomando de pip para instalar paquetes | Prepara las bibliotecas importadas por el proyecto y las pruebas |
| `-r` | Lee los requisitos desde un archivo | Evita escribir y mantener las dependencias a mano en el comando |
| `requirements-linux.txt` / `requirements.txt` | Archivo que sigue a `-r` | Selecciona la lista del sistema correspondiente; no es el nombre de un paquete |

En el archivo Linux, `-r backend/requirements.txt` y `-r frontend/requirements.txt` incluyen las listas de ambos módulos. Una línea `paquete==versión` fija una versión; `>=7,<8` expresa un intervalo; una condición como `; sys_platform == 'win32'` solo aplica en Windows. Estos signos pertenecen al formato del archivo, no son argumentos extra de la terminal.

`unittest`, `venv`, `pathlib`, `subprocess` y `sys` son parte de Python: no requieren `pip install unittest` ni paquetes añadidos. Las dependencias pueden descargarse en este paso; las pruebas RF01–RF04 descritas no necesitan conectarse a Supabase.

### 4. Revisar Tkinter únicamente si falta

```bash
python -c "import tkinter; print(tkinter.TkVersion)"
```

Si imprime una versión, no instalarlo de nuevo. Si falla el import, consultar la explicación Linux/Windows al final de la guía. Para el Python de Fedora:

```bash
sudo dnf install python3-tkinter
```

`sudo` pide privilegios administrativos; `dnf` es el gestor de paquetes de Fedora; `install` solicita instalar y `python3-tkinter` identifica el paquete del sistema. En Debian/Ubuntu la alternativa es `sudo apt install python3-tk`: `apt` sustituye a dnf y el paquete tiene otro nombre. Elegir el comando de la distribución, no ambos. No se utiliza `-y` para que el gestor muestre los cambios y solicite su confirmación habitual.

La biblioteca se importa en las pruebas frontend aunque no abran ventanas. `python -m tkinter` sirve para abrir una ventana de diagnóstico; necesita una sesión gráfica y no ejecuta pruebas de Fixify.

### 5. Ejecutar cada suite completa desde la raíz

```bash
python -m unittest discover -s backend/tests -v
python -m unittest discover -s frontend/tests -v
```

| Parte | Significado | Por qué aparece |
| --- | --- | --- |
| `python` | Intérprete del entorno | Debe tener instaladas las dependencias anteriores |
| `-m` | Ejecutar un módulo | Se usa el runner de ese Python sin necesitar un ejecutable externo |
| `unittest` | Framework y runner incluido en Python | Descubre y ejecuta las clases/métodos de pruebas del repositorio |
| `discover` | Modo de búsqueda de archivos | Permite ejecutar un conjunto sin escribir cada clase o método |
| `-s` | Directorio inicial de búsqueda, abreviación de `--start-directory` | Limita el descubrimiento a backend o frontend y permite importarlos con su organización actual |
| `backend/tests` / `frontend/tests` | Valor que recibe `-s` | Ruta relativa a la raíz; `-s` necesita este valor inmediatamente después |
| `-v` | Salida detallada, abreviación de `--verbose` | Muestra el nombre y resultado de cada método para localizar el que falla; no añade casos ni cambia las reglas verificadas |

Al no especificar `-p`, discover usa el patrón predeterminado `test*.py`. Dentro de los archivos identifica clases de unittest y métodos con prefijo `test`. El backend contiene actualmente 15 casos y el frontend 24. Revisar el total: `Ran 0 tests` no valida ninguna funcionalidad, aunque el proceso termine sin errores.

Se indican dos comandos porque las pruebas usan carpetas y ajustes de importación específicos. No sustituirlos por `python -m unittest` desde la raíz esperando que descubra automáticamente el mismo conjunto.

### 6. Ejecutar un archivo concreto

```bash
python -m unittest discover -s backend/tests -p test_clients.py -v
```

Conserva el mismo runner, directorio inicial y salida detallada. `-p` es la abreviación de `--pattern`: el valor `test_clients.py` es un patrón de nombre de archivo. En este caso, al no tener comodines, selecciona solamente ese archivo. No escribir `backend/tests/test_clients.py` después de `-p`: la carpeta ya está indicada mediante `-s`.

Para varios archivos con un patrón, ponerlo entre comillas para que lo interprete unittest y no lo expanda el shell:

```bash
python -m unittest discover -s backend/tests -p "test_*.py" -v
```

Aquí `*` representa cualquier secuencia de caracteres entre `test_` y `.py`. Las comillas no forman parte del patrón recibido. En las secciones de cada archivo se da el comando exacto para evitar seleccionar casos de más.

### 7. Ejecutar una clase o un método concreto

Desde la raíz, entrar primero en la carpeta de pruebas:

```bash
cd backend/tests
python -m unittest test_clients.ClientTests.test_client_with_equipment_is_preserved -v
cd ../..
```

| Parte | Significado | Motivo |
| --- | --- | --- |
| `cd backend/tests` | Cambia la carpeta de la terminal | Hace importable `test_clients` con los ajustes actuales del proyecto |
| `test_clients` | Nombre del módulo Python, sin `.py` | Identifica el archivo que contiene la prueba |
| `ClientTests` | Clase de unittest dentro de ese módulo | Elige el grupo de casos |
| `test_client_with_equipment_is_preserved` | Método específico de la clase | Ejecuta solo la protección del cliente con equipo |
| Puntos entre nombres | Separadores de un nombre Python calificado | No son separadores de carpetas |
| `-v` | Muestra nombre y resultado | Confirma qué caso se ejecutó |
| `cd ../..` | Sube dos carpetas | Vuelve a la raíz, desde tests → backend → Fixify, para que los comandos siguientes resuelvan sus rutas |

Este modo no lleva `discover`, `-s` ni `-p`: ya se especifica directamente el objeto importable. Para toda la clase, usar `python -m unittest test_clients.ClientTests -v`; para todo el módulo, `python -m unittest test_clients -v`. En frontend el mismo esquema utiliza, por ejemplo, `test_models.ModelsTests.test_strict_types_and_required_fields` desde `frontend/tests`.

Como alternativa sin cambiar de carpeta, filtrar desde discover:

```bash
python -m unittest discover -s backend/tests -p test_clients.py -k test_client_with_equipment_is_preserved -v
```

`-k` selecciona métodos cuyo nombre completo coincida con el filtro. Sin comodines, busca esa cadena como subcadena. Aquí se combina con `-p` para limitar primero el archivo y después el método. Confirmar en la salida que se ejecutó exactamente el caso deseado; una cadena más corta podría seleccionar varios casos. [Opciones oficiales de unittest](https://docs.python.org/3/library/unittest.html#command-line-interface).

### 8. Entender el resultado y terminar

Una ejecución aprobada termina con `OK` y código de salida 0. `FAIL` indica una aserción incumplida; `ERROR` una excepción inesperada o problema de importación/configuración. Los fallos se acompañan de un código distinto de cero. Los ejemplos de diagnóstico y las expectativas por método se desarrollan después de los comandos por archivo.

Para detener la suite ante el primer fallo, añadir `-f` (`--failfast`):

```bash
python -m unittest discover -s backend/tests -v -f
```

Es útil mientras se corrige el primer problema. Para la validación final ejecutar sin `-f`, de modo que se conozca el resultado de todos los casos. No usarlo para afirmar que la suite completa pasó si se detuvo antes de terminar.

Si el entorno se activó con un script, `deactivate` restaura la terminal al finalizar. No borra el venv. Si se utilizó directamente su ejecutable no hay nada que desactivar.

Fuentes de los comandos: [venv](https://docs.python.org/3/library/venv.html), [pip install](https://pip.pypa.io/en/stable/cli/pip_install/) y [unittest](https://docs.python.org/3/library/unittest.html).


## Ejecutar desde este notebook

Abrir el notebook no ejecuta comandos. Seleccionar un kernel que use el entorno del proyecto; activar un venv en una terminal **no cambia automáticamente el kernel ya abierto**. Ejecutar primero la celda de preparación que define `ROOT` y `run_tests`; después elegir una celda de suite o archivo. Run All repite las suites completas y los archivos individuales que contiene la guía: esos casos se ejecutarían más de una vez.

### Qué hace la celda de preparación

| Expresión | Función y motivo |
| --- | --- |
| `Path.cwd()` | Obtiene la carpeta de trabajo del kernel, que puede diferir de la ubicación del archivo notebook |
| `ROOT / "backend/tests"` | Compone una ruta sin depender del separador de Windows o Linux |
| `.is_dir()` | Comprueba que esa carpeta existe antes de lanzar comandos |
| `Path.cwd().parents` | Recorre las carpetas superiores hasta encontrar la raíz, incluso al abrir desde docs/Test |
| `assert ...` | Interrumpe con una explicación si no se encuentra la estructura esperada |
| `sys.executable` | Usa el Python del kernel, evitando llamar a otro `python` de PATH |
| `*arguments` | Recibe una cantidad variable de argumentos, por ejemplo discover, -s, carpeta y -v |
| `[sys.executable, "-m", "unittest", *arguments]` | Construye el comando como lista de argumentos, sin concatenar una cadena para el shell |
| `subprocess.run(...)` | Ejecuta unittest en un proceso hijo y espera su resultado; el detalle aparece en la salida que muestre el entorno de Jupyter |
| `cwd=ROOT` | Hace que todas las rutas relativas del comando se resuelvan desde la raíz del proyecto |
| `result.returncode` | Consulta el código de salida: 0 indica ejecución aprobada |
| `raise RuntimeError(...)` | Marca la celda como fallida si unittest termina con error, en lugar de presentar silenciosamente el comando como correcto |

Por ejemplo:

```python
run_tests("discover", "-s", "backend/tests", "-p", "test_clients.py", "-v")
```

Equivale a `<Python del kernel> -m unittest discover -s backend/tests -p test_clients.py -v`, con la raíz como directorio de trabajo. Cada string representa un argumento completo; no poner el comando entero dentro de un único string.

Verificar el kernel con `import sys; print(sys.executable)` en una celda. Si `run_tests` no está definido, ejecutar la celda de preparación. Si el helper no encuentra la raíz, iniciar Jupyter desde Fixify o una de sus subcarpetas, por ejemplo docs/Test; no corregirlo cambiando las rutas de las pruebas al azar.


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = next((directory for directory in [Path.cwd(), *Path.cwd().parents]
             if (directory / "backend/tests").is_dir()
             and (directory / "frontend/tests").is_dir()), None)
assert ROOT is not None, "Abra el notebook dentro del repositorio Fixify"

def run_tests(*arguments):
    result = subprocess.run([sys.executable, "-m", "unittest", *arguments], cwd=ROOT)
    if result.returncode:
        raise RuntimeError(f"Pruebas fallidas: código {result.returncode}")


## Pruebas de backend

```bash
python -m unittest discover -s backend/tests -v
```

**Lectura del comando:** `discover` carga los archivos de prueba desde `-s backend/tests`. Sin `-p`, utiliza `test*.py`; `-v` permite ver todos los casos y sus resultados. Usar esta ejecución como regresión completa del componente después de corregir un caso aislado.


In [ ]:
run_tests("discover", "-s", "backend/tests", "-v")


### test_schema_contract.py

Columnas/FK físicas del SQL, bcrypt, login y activación de usuarios.

```bash
python -m unittest discover -s backend/tests -p test_schema_contract.py -v
```

**Lectura del comando:** `python -m unittest` ejecuta el runner del entorno; `discover` busca pruebas; `-s backend/tests` fija la carpeta inicial; `-p test_schema_contract.py` limita la búsqueda a este archivo y `-v` muestra cada caso. Usar este comando para revisar el módulo sin repetir toda la suite. La celda siguiente pasa esos mismos argumentos a `run_tests`, usando el Python del kernel.


In [ ]:
run_tests("discover", "-s", "backend/tests", "-p", "test_schema_contract.py", "-v")


### test_equipment.py

Alta/listado, propietario inexistente, permisos, JWT previo tras cambio de rol/desactivación, serie duplicada, edición y eliminación con historial.

```bash
python -m unittest discover -s backend/tests -p test_equipment.py -v
```

**Lectura del comando:** `python -m unittest` ejecuta el runner del entorno; `discover` busca pruebas; `-s backend/tests` fija la carpeta inicial; `-p test_equipment.py` limita la búsqueda a este archivo y `-v` muestra cada caso. Usar este comando para revisar el módulo sin repetir toda la suite. La celda siguiente pasa esos mismos argumentos a `run_tests`, usando el Python del kernel.


In [ ]:
run_tests("discover", "-s", "backend/tests", "-p", "test_equipment.py", "-v")


### test_clients.py

Alta/búsqueda, datos inválidos, edición, teléfono repetido, eliminación y conservación de clientes con equipos.

```bash
python -m unittest discover -s backend/tests -p test_clients.py -v
```

**Lectura del comando:** `python -m unittest` ejecuta el runner del entorno; `discover` busca pruebas; `-s backend/tests` fija la carpeta inicial; `-p test_clients.py` limita la búsqueda a este archivo y `-v` muestra cada caso. Usar este comando para revisar el módulo sin repetir toda la suite. La celda siguiente pasa esos mismos argumentos a `run_tests`, usando el Python del kernel.


In [ ]:
run_tests("discover", "-s", "backend/tests", "-p", "test_clients.py", "-v")


### Ejecutar un caso individual de backend

Desde una terminal en la raíz:

```bash
cd backend/tests
python -m unittest test_schema_contract.SchemaTests.test_create_and_authenticate_sql_user -v
```

Sustituir módulo, clase y método por el caso que se quiera repetir. Un resultado correcto termina en `OK`; `FAIL` indica una aserción incumplida y `ERROR` una excepción/configuración faltante. Leer el traceback y repetir el caso después de corregirlo.

Después de ejecutar desde `backend/tests` o `frontend/tests`, use `cd ../..` para volver a la raíz antes de lanzar otros comandos de esta guía. El nombre con puntos identifica módulo, clase y método; no es una ruta de archivo.


## Pruebas de frontend

```bash
python -m unittest discover -s frontend/tests -v
```

**Lectura del comando:** `discover` carga los archivos de prueba desde `-s frontend/tests`. Sin `-p`, utiliza `test*.py`; `-v` permite ver todos los casos y sus resultados. Usar esta ejecución como regresión completa del componente después de corregir un caso aislado.


In [ ]:
run_tests("discover", "-s", "frontend/tests", "-v")


### test_models.py

Contratos estrictos, respuestas malformadas y datos inválidos antes de enviar HTTP.

```bash
python -m unittest discover -s frontend/tests -p test_models.py -v
```

**Lectura del comando:** `python -m unittest` ejecuta el runner del entorno; `discover` busca pruebas; `-s frontend/tests` fija la carpeta inicial; `-p test_models.py` limita la búsqueda a este archivo y `-v` muestra cada caso. Usar este comando para revisar el módulo sin repetir toda la suite. La celda siguiente pasa esos mismos argumentos a `run_tests`, usando el Python del kernel.


In [ ]:
run_tests("discover", "-s", "frontend/tests", "-p", "test_models.py", "-v")


### test_async_frontend.py

Rutas HTTP, errores, limpieza de sesión, tareas concurrentes y callbacks en el hilo principal.

```bash
python -m unittest discover -s frontend/tests -p test_async_frontend.py -v
```

**Lectura del comando:** `python -m unittest` ejecuta el runner del entorno; `discover` busca pruebas; `-s frontend/tests` fija la carpeta inicial; `-p test_async_frontend.py` limita la búsqueda a este archivo y `-v` muestra cada caso. Usar este comando para revisar el módulo sin repetir toda la suite. La celda siguiente pasa esos mismos argumentos a `run_tests`, usando el Python del kernel.


In [ ]:
run_tests("discover", "-s", "frontend/tests", "-p", "test_async_frontend.py", "-v")


### test_module_integration.py

Descubrimiento de servicios/vistas y navegación por rol.

```bash
python -m unittest discover -s frontend/tests -p test_module_integration.py -v
```

**Lectura del comando:** `python -m unittest` ejecuta el runner del entorno; `discover` busca pruebas; `-s frontend/tests` fija la carpeta inicial; `-p test_module_integration.py` limita la búsqueda a este archivo y `-v` muestra cada caso. Usar este comando para revisar el módulo sin repetir toda la suite. La celda siguiente pasa esos mismos argumentos a `run_tests`, usando el Python del kernel.


In [ ]:
run_tests("discover", "-s", "frontend/tests", "-p", "test_module_integration.py", "-v")


### test_window_focus.py

Enfoque de la ventana tras login y cierre.

```bash
python -m unittest discover -s frontend/tests -p test_window_focus.py -v
```

**Lectura del comando:** `python -m unittest` ejecuta el runner del entorno; `discover` busca pruebas; `-s frontend/tests` fija la carpeta inicial; `-p test_window_focus.py` limita la búsqueda a este archivo y `-v` muestra cada caso. Usar este comando para revisar el módulo sin repetir toda la suite. La celda siguiente pasa esos mismos argumentos a `run_tests`, usando el Python del kernel.


In [ ]:
run_tests("discover", "-s", "frontend/tests", "-p", "test_window_focus.py", "-v")


### Ejecutar un caso individual de frontend

Desde una terminal en la raíz:

```bash
cd frontend/tests
python -m unittest test_models.ModelsTests.test_strict_types_and_required_fields -v
```

Sustituir módulo, clase y método por el caso que se quiera repetir. Un resultado correcto termina en `OK`; `FAIL` indica una aserción incumplida y `ERROR` una excepción/configuración faltante. Leer el traceback y repetir el caso después de corregirlo.

Después de ejecutar desde `backend/tests` o `frontend/tests`, use `cd ../..` para volver a la raíz antes de lanzar otros comandos de esta guía. El nombre con puntos identifica módulo, clase y método; no es una ruta de archivo.



## Comprobación manual del frontend

Arrancar API y frontend siguiendo sus README. Verificar login válido/inválido; menú por rol; crear cliente; editarlo; buscarlo; registrar equipo con ese cliente; editar el equipo; comprobar que eliminar el cliente con equipo devuelve conflicto; eliminar un equipo sin órdenes y después el cliente libre. Confirmar cancelar formulario y cancelar eliminación. En usuarios, cambiar rol/desactivar y comprobar que un token anterior pierde permisos.

## Evidencia de ejecución

Las salidas de unittest muestran cada caso, total, tiempo y resultado. Conservar la fecha, versiones y salida al registrar una nueva entrega. Este notebook contiene instrucciones; no presenta celdas sin ejecutar como resultados aprobados.


## Contratos de edición y eliminación

`frontend/tests/test_crud_services.py` verifica PUT/DELETE, campos opcionales, rechazo local de entradas inválidas y errores 409 por historial.

```bash
python -m unittest discover -s frontend/tests -p test_crud_services.py -v
```

**Lectura del comando:** `python -m unittest` ejecuta el runner del entorno; `discover` busca pruebas; `-s frontend/tests` fija la carpeta inicial; `-p test_crud_services.py` limita la búsqueda a este archivo y `-v` muestra cada caso. Usar este comando para revisar el módulo sin repetir toda la suite. La celda siguiente pasa esos mismos argumentos a `run_tests`, usando el Python del kernel.


In [ ]:
run_tests("discover", "-s", "frontend/tests", "-p", "test_crud_services.py", "-v")


## Cómo funcionan las pruebas y para qué sirven

Esta ampliación conserva las instrucciones anteriores. Describe los **39 casos actuales: 15 backend y 24 frontend**. Una prueba automatizada ejecuta una acción con datos controlados y compara el resultado con una expectativa. Sirve para detectar si un cambio rompe un comportamiento que ya estaba funcionando; aprobarla demuestra las comprobaciones escritas, no todas las posibilidades del sistema.

### Ciclo de ejecución

1. `unittest discover` importa los archivos `test_*.py`, identifica las clases de pruebas y ejecuta sus métodos `test_*`.
2. `setUp()` prepara recursos nuevos antes de cada método cuando la clase lo define. En los servicios asíncronos se utiliza `asyncSetUp()`; `IsolatedAsyncioTestCase` ejecuta cada caso con un ciclo de eventos aislado.
3. El caso realiza una acción: enviar una petición, validar un modelo, cambiar el rol de una cuenta o cancelar una tarea.
4. Las aserciones comprueban el resultado. `assertEqual` compara valores; `assertTrue` comprueba una condición; `assertRaises` exige una excepción prevista. Que una entrada inválida produzca el error esperado es un **caso aprobado**.
5. `tearDown()`, `asyncTearDown()` o `finally` cierran conexiones, clientes HTTP y tareas para que el caso siguiente no dependa del anterior.

Los nombres de los métodos aparecen en la salida con `-v`. Un método puede incluir varias comprobaciones; por eso `Ran 39 tests` no significa solo 39 aserciones. Backend y frontend se ejecutan por separado: la cifra combinada es la suma de ambas suites.

### Qué se ejecuta realmente y qué se simula

| Grupo | Componentes reales | Recursos controlados | Utilidad |
| --- | --- | --- | --- |
| Backend: clientes/equipos | Routers FastAPI, dependencias de permisos, Pydantic, repositorios SQLAlchemy y JWT | Aplicación de pruebas, SQLite en memoria y cuentas locales | Detecta fallos del flujo HTTP → validación → permisos → persistencia |
| Backend: contrato SQL | Modelos ORM y archivo SQL del repositorio | Comparación local de nombres, tipos acotados, nulabilidad y relaciones | Detecta desalineación del modelo respecto al esquema declarado |
| Frontend: servicios/modelos | ApiClient, Services y contratos Pydantic | Respuestas definidas con `httpx.MockTransport` | Verifica solicitudes y manejo de respuestas sin depender de un servidor |
| Frontend contra router real | Servicio de equipos y router real | `httpx.ASGITransport` dirige las peticiones a la aplicación en memoria | Comprueba que ambos lados entienden el mismo contrato |
| Frontend: runner | AsyncRunner y su hilo/ciclo de eventos | `FakeRoot` y `Owner` sustituyen widgets | Verifica concurrencia, callbacks y cierre sin abrir ventanas |
| Frontend: descubrimiento/enfoque | Funciones de descubrimiento y métodos de ventana | `patch`, módulos artificiales y `Mock` | Verifica decisiones y llamadas sin necesitar una GUI real |

`MockTransport` devuelve las respuestas definidas por el test: no demuestra que el backend aplique esas reglas. `ASGITransport` y `TestClient` sí ejecutan los routers y dependencias de la aplicación de pruebas, sin levantar Uvicorn ni contactar Supabase.

### Preparación de la base temporal

`EquipmentTests.setUp()` crea un motor `sqlite://` y usa `StaticPool` para compartir la conexión en memoria entre las sesiones del caso. `check_same_thread=False` permite utilizarla desde los hilos del cliente de pruebas. Activa claves foráneas mediante `PRAGMA foreign_keys=ON`.

Las tablas de clientes y equipos se crean con SQL explícito y los nombres físicos del proyecto: `id_cliente`, `id_equipo` y `falla_reportada`. Esto permite detectar un ORM que apunte a otra columna. Se añaden tablas de órdenes, roles y usuarios; se prepara el cliente ID 7 y cuentas admin, recepción y técnico con hashes bcrypt. El JWT de prueba tiene expiración y corresponde a una cuenta existente.

`dependency_overrides[get_db]` entrega sesiones de ese motor a FastAPI. `ClientTests` reutiliza esta preparación y añade el router de clientes. Cada método recibe una base nueva. Las variables de conexión/clave se fijan para la prueba antes de importar la configuración; **no hace falta DataBase.env ni modificar datos reales**.

### Ejemplo de lectura de una prueba

En `test_client_with_equipment_is_preserved`, el cliente ID 7 ya existe. Primero se registra un equipo suyo; después se intenta eliminarlo. El test exige HTTP 409 y comprueba que el cliente sigue en el listado. Así detecta una eliminación que dejara al equipo sin propietario. La regla se comprueba en el backend real de pruebas, no solo ocultando un botón.

En `test_current_account_state_and_role`, se emite un JWT de administrador y después se cambia su cuenta a técnico. La misma cabecera debe obtener 403 al consultar equipos. Tras desactivar la cuenta debe obtener 401. Esto comprueba que los permisos se leen de la cuenta actual y no se conservan por el rol antiguo del token.


## Detalle de `backend/tests/test_schema_contract.py`

### `test_actual_account_state_column`

**Cómo funciona:** Inspecciona el mapeo de User.is_active sin enviar HTTP.

**Qué comprueba:** La columna física se llama is_active y no existe is_activate en los metadatos.

**Para qué sirve:** Evita repetir un error de columna inexistente al consultar usuarios.

### `test_all_sql_columns_and_foreign_keys`

**Cómo funciona:** Lee BASE DE DATOS FIXIFY.sql, extrae CREATE TABLE y compara las tablas registradas en Base.metadata.

**Qué comprueba:** Coinciden tablas, columnas, nulabilidad, longitudes VARCHAR y destinos de claves foráneas.

**Para qué sirve:** Detecta divergencias entre SQL y ORM; no compara una base PostgreSQL desplegada ni todos los detalles posibles de SQL.

### `test_create_and_authenticate_sql_user`

**Cómo funciona:** Crea SQLite y un rol; registra una cuenta mediante el repositorio; prueba el hash y el router de login antes/después de desactivar y reactivar.

**Qué comprueba:** Hash válido para la contraseña correcta; inválido para otra. Login 200 con rol recepción, contraseña incorrecta 401 y cuenta inactiva 403; reactivarla permite 200. IDs inexistentes devuelven None en el repositorio.

**Para qué sirve:** Valida RF01 y el ciclo de activación sin borrar la cuenta. No recorre todos los endpoints de administración de usuarios.



## Detalle de `backend/tests/test_equipment.py`

### `test_create_and_list_against_diagram_columns`

**Cómo funciona:** Envía POST con el equipo del cliente 7, consulta GET y lee directamente la fila SQL guardada.

**Qué comprueba:** POST 201, EQ-001, GET 200 con datos originales y la fila física (1, 7, No enciende); FK hacia id_cliente.

**Para qué sirve:** Comprueba RF04 de punta a punta y los aliases entre nombres API y SQL.

### `test_missing_client_and_invalid_payload`

**Cómo funciona:** Intenta registrar con client_id 999 y luego con tipo vacío.

**Qué comprueba:** 404 para propietario inexistente, 422 para campos inválidos y listado vacío.

**Para qué sirve:** Evita registros huérfanos y altas parciales tras entradas rechazadas.

### `test_optional_serial_and_order`

**Cómo funciona:** Registra un equipo sin serie y otro con SEGUNDO.

**Qué comprueba:** Ambos reciben 201; el listado devuelve IDs [1, 2] y serie null para el primero.

**Para qué sirve:** Permite dispositivos sin serie y mantiene orden estable de consulta.

### `test_permissions`

**Cómo funciona:** Intenta GET y POST sin JWT, con técnico y con administrador.

**Qué comprueba:** 401 sin sesión; 403 para técnico; 200/201 para administrador según operación.

**Para qué sirve:** Comprueba RF02 en el backend, independientemente del menú visible.

### `test_edit_delete_and_history`

**Cómo funciona:** Registra equipo, edita modelo/serie, intenta editar con técnico y añade una orden; intenta borrar. Luego retira la orden solo en la base de pruebas y vuelve a borrar.

**Qué comprueba:** PUT 200 con modelo actualizado; técnico 403; DELETE con orden 409; sin orden 200 y segundo DELETE 404.

**Para qué sirve:** Valida edición y conservación de historial. Retirar la orden es preparación del escenario siguiente, no una acción disponible en la interfaz.

### `test_edit_rejects_duplicate_and_unknown_owner`

**Cómo funciona:** Crea dos equipos con series distintas; intenta editar el primero usando la serie del segundo y luego un propietario inexistente.

**Qué comprueba:** 409 por serie ocupada y 404 por cliente inexistente.

**Para qué sirve:** Evita que la edición permita inconsistencias que el alta rechaza.

### `test_duplicate_serial`

**Cómo funciona:** Envía dos altas con la misma serie y consulta el listado.

**Qué comprueba:** Primera alta 201, segunda 409; solo queda un equipo.

**Para qué sirve:** Evita duplicados en altas sucesivas; no demuestra unicidad frente a peticiones simultáneas.

### `test_current_account_state_and_role`

**Cómo funciona:** Conserva un JWT de admin; cambia el rol de su cuenta en SQLite a técnico y después la desactiva.

**Qué comprueba:** El mismo token obtiene 403 después del cambio y 401 después de desactivar.

**Para qué sirve:** Detecta permisos obsoletos almacenados exclusivamente en el JWT.

### `test_frontend_service_consumes_real_router`

**Cómo funciona:** Conecta ApiClient a la aplicación real del caso mediante ASGITransport, añade el JWT e invoca crear/listar desde Services.

**Qué comprueba:** El servicio interpreta EQ-001 y devuelve modelos cuyos datos coinciden con el alta.

**Para qué sirve:** Comprueba compatibilidad real entre el servicio frontend de equipos y el backend sin red externa.



## Detalle de `backend/tests/test_clients.py`

### `test_client_registration_search_and_validation`

**Cómo funciona:** Registra Ana con espacios en el nombre; busca por Ana; repite teléfono; envía nombre vacío, teléfono alfabético, correo inválido y alta con técnico.

**Qué comprueba:** 201 con ID positivo y nombre recortado; búsqueda encuentra el registro; duplicado 400, entradas inválidas 422 y técnico 403.

**Para qué sirve:** Valida RF03 y los errores que debe mostrar la interfaz.

### `test_edit_delete_and_related_equipment`

**Cómo funciona:** Crea un cliente y cambia su nombre; intenta usar el teléfono del cliente 7; intenta eliminar con técnico. Elimina clientes libres e intenta editar uno eliminado.

**Qué comprueba:** Edición 200 con nombre corregido; teléfono ocupado 409; técnico 403; bajas permitidas 200 y edición posterior 404.

**Para qué sirve:** Comprueba actualización y baja de registros libres. Aunque el nombre menciona equipos, este caso no crea equipos; esa protección se prueba en el siguiente.

### `test_client_with_equipment_is_preserved`

**Cómo funciona:** Registra un equipo del cliente 7 e intenta eliminar al propietario.

**Qué comprueba:** 409 y el cliente permanece en el directorio.

**Para qué sirve:** Impide perder el propietario de un equipo existente.



## Detalle de `frontend/tests/test_models.py`

### `test_strict_types_and_required_fields`

**Cómo funciona:** Construye usuarios con ID string/bool, estado string, rol desconocido y cliente sin teléfono.

**Qué comprueba:** Cada entrada inválida levanta ValidationError.

**Para qué sirve:** Evita conversiones silenciosas y datos incompletos en los contratos públicos.

### `test_extras_optional_fields_and_secret_repr`

**Cómo funciona:** Carga un usuario con hashed_password extra, un cliente sin correo y modelos de login/sesión con secretos.

**Qué comprueba:** El hash extra se ignora, correo es None y repr no contiene contraseña ni token.

**Para qué sirve:** Reduce exposición de secretos en representaciones de modelos; no audita todos los logs.

### `test_views_render_models`

**Cómo funciona:** Invoca row de clientes, equipos y usuarios con modelos tipados, sin construir ventanas.

**Qué comprueba:** Tuplas en el orden esperado, guiones para opcionales, EQ-003 y estado Inactivo.

**Para qué sirve:** Detecta errores de adaptación a la tabla; no evalúa colores o tamaños.

### `test_invalid_record_rejected_before_view`

**Cómo funciona:** MockTransport devuelve listas con campos faltantes o ID booleano.

**Qué comprueba:** El servicio levanta ApiError relacionado con el contrato Cliente.

**Para qué sirve:** Impide entregar registros malformados a la vista.

### `test_valid_list_returns_models`

**Cómo funciona:** El transporte responde con un cliente válido sin correo.

**Qué comprueba:** El resultado contiene una instancia Cliente con ID 7.

**Para qué sirve:** Comprueba la conversión JSON a modelos que consume la UI; el caso actual no envía una lista vacía.

### `test_invalid_input_is_not_sent_and_no_secret_leaks`

**Cómo funciona:** Intenta crear usuario con nombre demasiado corto; registra cualquier petición recibida por el transporte.

**Qué comprueba:** ApiError sin contraseña en el mensaje, sin causa encadenada visible y lista de peticiones vacía.

**Para qué sirve:** Detecta validación tardía y filtración del secreto en ese error.

### `test_malformed_creation_confirmation_rejected`

**Cómo funciona:** El transporte devuelve 201 y solo mensaje, sin folio ni detalles del equipo.

**Qué comprueba:** ApiError por contrato EquipoRegistrado.

**Para qué sirve:** Evita tratar una confirmación incompleta como un alta válida.



## Detalle de `frontend/tests/test_async_frontend.py`

### `test_login_form_and_session_cleanup`

**Cómo funciona:** Prepara un token anterior; inicia login con espacios en la contraseña; asigna la nueva sesión y después la limpia.

**Qué comprueba:** Login no lleva Authorization y conserva espacios codificados; login por sí solo no cambia el token; consultas protegidas lo incluyen hasta limpiarlo.

**Para qué sirve:** Verifica el contrato de sesión y evita alterar la contraseña capturada.

### `test_module_routes_and_payloads`

**Cómo funciona:** Invoca altas, búsqueda, listado y cambios de rol/estado sobre el transporte que registra peticiones.

**Qué comprueba:** Correo/serie vacíos se convierten a null; client_id conserva tipo; q conserva Ana & Luis; PATCH usa las rutas de rol y habilitar/inhabilitar.

**Para qué sirve:** Detecta errores de URL, método y serialización sin depender de un servidor.

### `test_errors_and_invalid_responses`

**Cómo funciona:** Simula 403/422 con detail, 502 con HTML, 200 con JSON inválido y ReadTimeout.

**Qué comprueba:** ApiError conserva código HTTP cuando existe y mensajes legibles; timeout indica que la petición tardó demasiado.

**Para qué sirve:** Evita excepciones sin contexto o mensajes incomprensibles en la UI.

### `test_invalid_list_and_login`

**Cómo funciona:** Devuelve un objeto inesperado en lugar de una lista o sesión válida.

**Qué comprueba:** ApiError por lista inválida y por contrato Sesion.

**Para qué sirve:** Evita llenar tablas o aceptar sesiones con respuestas incompatibles.

### `test_dashboard_queries_are_concurrent`

**Cómo funciona:** Tres consultas simuladas esperan un Event que solo se activa cuando las tres arrancan; una falla deliberadamente.

**Qué comprueba:** La consulta fallida retorna como ApiError y las otras dos conservan sus datos.

**Para qué sirve:** Detecta ejecución secuencial que bloquearía la espera y pérdida de métricas por un fallo independiente.

### `test_callback_runs_on_main_thread`

**Cómo funciona:** AsyncRunner ejecuta una coroutine que devuelve su ID de hilo; FakeRoot bombea los callbacks.

**Qué comprueba:** submit retorna sin esperar; trabajo corre en otro hilo y callback en el hilo principal.

**Para qué sirve:** Protege el acceso a widgets Tk desde el hilo correcto sin probar una ventana real.

### `test_cancel_owner_and_destroyed_view`

**Cómo funciona:** Cancela tareas de un Owner y termina otra tarea cuyo propietario indica que ya no existe.

**Qué comprueba:** Future cancelado para el primero; sin resultados entregados a ninguno; pendientes liberados.

**Para qué sirve:** Evita callbacks tardíos sobre vistas cerradas.

### `test_close_cancels_work_and_closes_http`

**Cómo funciona:** Inicia tarea larga y llama close con coroutine de limpieza y callback de cierre.

**Qué comprueba:** Tarea cancelada y orden de limpieza [http, tk]; tearDown comprueba hilo terminado y loop cerrado.

**Para qué sirve:** Detecta cierre incompleto o destrucción de la interfaz antes de liberar sus recursos.



## Detalle de `frontend/tests/test_module_integration.py`

### `test_existing_services_and_menus`

**Cómo funciona:** Construye Services con una API compartida y consulta los menús de los tres roles.

**Qué comprueba:** Los cuatro servicios conservan la misma API; admin ve Dashboard/Equipos/Clientes/Usuarios, recepción los tres primeros y técnico Dashboard.

**Para qué sirve:** Detecta cambios accidentales de registro o navegación; no sustituye permisos backend.

### `test_new_service_and_view_need_no_manual_registration`

**Cómo funciona:** patch simula descubrimiento/importación de un servicio y vista Orders creados en memoria.

**Qué comprueba:** Services encuentra orders; la vista aparece para admin y no para técnico.

**Para qué sirve:** Comprueba extensibilidad del mecanismo sin implementar órdenes reales.

### `test_request_sets_owner_and_default_error_callback`

**Cómo funciona:** Sustituye runner por submit controlado y ejecuta una operación que duplica 3.

**Qué comprueba:** Resultado 6, propietario correcto, callback de error de la vista y retorno future.

**Para qué sirve:** Verifica que AsyncFrame conecta operaciones con la vista correcta.

### `test_duplicate_service_key_is_rejected`

**Cómo funciona:** Simula dos clases con la misma clave orders dentro del módulo descubierto.

**Qué comprueba:** service_classes levanta ValueError que menciona duplicado.

**Para qué sirve:** Evita que un servicio sobrescriba a otro silenciosamente.

### `test_fetch_and_save_delegate_to_declared_service`

**Cómo funciona:** Usa un servicio artificial con listar y crear, e invoca los métodos base CollectionView.

**Qué comprueba:** fetch devuelve la lista del servicio y save entrega equipo_id 7.

**Para qué sirve:** Comprueba delegación reutilizable sin construir Tk ni duplicar consultas por vista.



## Detalle de `frontend/tests/test_window_focus.py`

### `test_raise_window_and_release_priority`

**Cómo funciona:** Mock con ventana existente recibe bring_to_front y después _release_topmost.

**Qué comprueba:** Se llama deiconify/lift/focus_force, se activa topmost y se programa liberación a los 250 ms; luego topmost pasa a False.

**Para qué sirve:** Detecta llamadas faltantes para mostrar la ventana tras login; el gestor de ventanas real se revisa manualmente.

### `test_closed_window_is_ignored`

**Cómo funciona:** El Mock indica que la ventana ya no existe y recibe bring_to_front.

**Qué comprueba:** lift no se ejecuta.

**Para qué sirve:** Evita intentar enfocar una ventana destruida.



## Detalle de `frontend/tests/test_crud_services.py`

### `test_edit_and_delete_contracts`

**Cómo funciona:** MockTransport registra PUT/DELETE y devuelve registros o mensajes; edita cliente/equipo con opcionales vacíos y elimina ambos.

**Qué comprueba:** IDs y modelos válidos, correo/serie None y secuencia exacta PUT clientes/4, PUT equipos/4, DELETE clientes/4, DELETE equipos/4.

**Para qué sirve:** Comprueba el contrato HTTP del CRUD, no sus reglas SQL.

### `test_invalid_edit_and_history_conflict`

**Cómo funciona:** Edita con nombre/teléfono inválidos y luego simula un 409 al eliminar equipo.

**Qué comprueba:** Edición rechazada sin ninguna petición; eliminación produce ApiError con el mensaje de órdenes asociadas.

**Para qué sirve:** Evita envíos inválidos y conserva la explicación del bloqueo para la interfaz.



## Interpretar un fallo y repetirlo

Ejemplo ilustrativo, no salida de una ejecución:

```text
FAIL: test_client_with_equipment_is_preserved (...)
AssertionError: 200 != 409
```

El caso esperaba bloquear la eliminación, pero recibió éxito. Revisar el endpoint y la consulta de relaciones; cambiar la expectativa a 200 ocultaría la pérdida de integridad. Si la salida dice `ERROR` y muestra `ModuleNotFoundError: tkinter`, el método no llegó a comprobar su regla: primero hay que reparar el entorno. Un `DeprecationWarning` puede aparecer junto a `OK`; no equivale por sí solo a una prueba fallida.

Para repetir solo el primer ejemplo, desde la raíz:

```bash
cd backend/tests
python -m unittest test_clients.ClientTests.test_client_with_equipment_is_preserved -v
```

Leer primero el nombre del método, después la última línea del traceback y finalmente las líneas que señalan la aserción. Comparar la preparación, acción y expectativa documentadas arriba. Después de corregir, repetir el caso y la suite del componente afectado.

### Qué no demuestran estas suites

No conectan al PostgreSQL remoto, no prueban migraciones desplegadas ni carreras de altas simultáneas. Tampoco validan visualmente la distribución de widgets, todas las rutas de usuarios, expiración/firma manipulada del JWT o todos los cambios de propietario con órdenes. No miden rendimiento bajo carga. Son límites de la cobertura actual, no afirmaciones de que esos comportamientos estén aprobados.

Las pruebas de interfaz usan modelos y sustitutos de widgets. Tkinter se importa, pero no abren ventanas: necesitar la biblioteca no significa necesitar un escritorio gráfico para esta suite. La comprobación manual anterior sí necesita una sesión gráfica.

Para una nueva entrega, conservar esta documentación y registrar requisito, escenario añadido, resultado y fecha. La evidencia debe incluir comando, versión de Python, cantidad de casos y resultado, sin credenciales. No confundir las celdas de instrucciones sin ejecutar con salidas de pruebas aprobadas.


## Tkinter en Linux y Windows 11

Fixify usa `ttkbootstrap`, que se apoya en `tkinter` y en las bibliotecas nativas Tcl/Tk. En Windows, el instalador tradicional de Python de python.org incluye por defecto la opción de Tcl/Tk e IDLE: normalmente ya está disponible al terminar la instalación. **Windows 11 por sí mismo no proporciona Tkinter**; depende de cómo se instaló Python. Una instalación personalizada que omita Tcl/Tk también puede necesitar añadir ese componente. [Documentación oficial del instalador de Windows](https://docs.python.org/3/using/windows.html).

En esta máquina Linux el intérprete no tenía `tkinter`. Distribuciones como Fedora y Debian/Ubuntu lo ofrecen como paquete separado; instalarlo solo hace falta si el Python elegido no lo incluye. Usar `python3-tkinter` en Fedora o `python3-tk` en Debian/Ubuntu para el Python de la distribución. Si se usa otro intérprete, el paquete debe corresponder a ese intérprete.

Por eso no se añade `python3-tkinter` a requirements.txt: esos archivos gestionan paquetes de pip, mientras que aquí falta un componente del intérprete y sus bibliotecas nativas. Crear un venv o instalar ttkbootstrap con pip no lo agrega al Python base.

Antes de instalar, comprobar desde el mismo entorno que ejecutará las pruebas:

```bash
python -c "import sys, tkinter; print(sys.executable); print(tkinter.TkVersion)"
```

Si funciona, no hace falta instalar Tkinter de nuevo. Para comprobar además una ventana real, usar `python -m tkinter` en una sesión gráfica. Si el import funciona pero aparece un error sobre DISPLAY al abrir la ventana, la biblioteca está instalada y falta acceso al entorno gráfico. [Documentación oficial de Tkinter](https://docs.python.org/3/library/tkinter.html).


## Pruebas de edición de usuarios — actualización 05-Oct-2026

La suite actual contiene **17 casos backend y 25 frontend**; las cifras anteriores se conservan como historial.

### backend/tests/test_users.py

```bash
python -m unittest discover -s backend/tests -p test_users.py -v
```

`python -m unittest` ejecuta el runner del entorno; discover busca casos; `-s backend/tests` indica carpeta; `-p test_users.py` selecciona solo el archivo; `-v` muestra cada resultado. No necesita servidor ni BD remota: reutiliza el escenario SQLite y las cuentas de EquipmentTests y registra el router real de usuarios.

- `test_edit_preserves_credentials_and_state`: edita nombre/rol como admin, verifica JSON completo y hash conservado. Desactiva, edita una cuenta inactiva sin reactivarla y vuelve a habilitar. Detecta cambios accidentales de credenciales/estado durante la edición.
- `test_edit_permissions_invalid_input_and_conflicts`: exige 401 sin sesión, 403 para recepción/técnico, 422 para nombre/rol inválidos, 409 para nombre duplicado y 404 para ID ausente; comprueba que el nombre original permanece tras los rechazos.

### frontend/tests/test_crud_services.py: test_user_edit_contract

```bash
python -m unittest discover -s frontend/tests -p test_crud_services.py -k test_user_edit_contract -v
```

El nuevo `-k` filtra por nombre de método dentro del archivo seleccionado. MockTransport registra una edición de usuario y devuelve un registro inactivo. Se comprueba PUT /usuarios/4, cuerpo username/role sin contraseña, modelo devuelto con ID conservado y estado False. Este caso valida el contrato del servicio; los casos backend anteriores validan persistencia y permisos reales.
